# Python-Based Behavioral & Customer Experience Analytics

This phase focuses on extending the SQL analytics layer into deeper behavioral, operational, and customer experience analysis using Python. While the SQL layer established core business metrics and operational KPIs, this section aims to uncover the underlying drivers influencing customer retention, satisfaction, and marketplace performance.

The analysis combines structured transactional data with unstructured customer feedback to simulate how modern ecommerce and quick-commerce analytics teams evaluate operational efficiency and customer experience at scale.

Key focus areas include:

- Customer segmentation based on repeat purchase behavior and spending patterns
- Retention and repeat purchase analysis
- Delivery experience and operational SLA analysis
- Category-level customer behavior trends
- AI-assisted thematic analysis of customer reviews using LLM workflows

The objective is not only to measure business performance, but also to identify operational and behavioral patterns that influence customer satisfaction, repeat purchases, and long-term marketplace growth.

This phase mirrors real-world business analyst workflows commonly used in consumer-tech and ecommerce environments, where SQL reporting is often augmented with Python-driven exploratory analysis and AI-assisted customer intelligence.

## SECTION 0 — Setup

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [0]:
df = spark.table(
    "workspace.default.final_quick_comm_dataset"
).toPandas()

In [0]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [0]:
df["delivery_variance_days"] = (
    df["order_delivered_customer_date"] -
    df["order_estimated_delivery_date"]
).dt.days

In [0]:
df["late_delivery_flag"] = (
    df["delivery_variance_days"] > 0
).astype(int)

In [0]:
df.head()

In [0]:
df.info()

## SECTION 1 — Customer Segmentation

In [0]:
customer_df = df.groupby(
    "customer_unique_id"
).agg({
    "order_id":"nunique",
    "payment_value":"sum",
    "review_score":"mean",
    "delivery_variance_days":"mean"
}).reset_index()

customer_df.columns = [
    "customer_unique_id",
    "total_orders",
    "total_spend",
    "avg_review_score",
    "avg_delivery_variance"
]

In [0]:
customer_df["customer_type"] = np.where(
    customer_df["total_orders"] > 1,
    "Repeat",
    "One-Time"
)

In [0]:
customer_summary = customer_df.groupby(
    "customer_type"
).agg({
    "total_orders":"mean",
    "total_spend":"mean",
    "avg_review_score":"mean",
    "avg_delivery_variance":"mean"
}).round(2)

customer_summary


- who spends more: One-Time
- who returns: 3.12% of total customers
- who churns: 

## SECTION 2 — Retention Analysis

In [0]:
df["cohort_month"] = df.groupby(
    "customer_unique_id"
)["order_purchase_timestamp"].transform("min").dt.to_period("M")

df["purchase_month"] = (
    df["order_purchase_timestamp"]
    .dt.to_period("M")
)

cohort_data = df.groupby([
    "cohort_month",
    "purchase_month"
]).agg({
    "customer_unique_id":"nunique"
}).reset_index()

cohort_pivot = cohort_data.pivot(
    index="cohort_month",
    columns="purchase_month",
    values="customer_unique_id"
)

cohort_pivot

## SECTION 3 — Delivery Experience Analysis

In [0]:
delivery_analysis = df.groupby(
    "late_delivery_flag"
).agg({
    "review_score":"mean",
    "payment_value":"mean",
    "order_id":"nunique"
}).round(2)

delivery_analysis

In [0]:
category_delivery = df.groupby(
    "product_category_name_english"
).agg({
    "late_delivery_flag":"mean",
    "review_score":"mean",
    "payment_value":"sum",
    "order_id":"nunique"
}).reset_index()

category_delivery.columns = [
    "category",
    "late_delivery_rate",
    "avg_review_score",
    "total_revenue",
    "total_orders"
]

category_delivery["late_delivery_rate"] *= 100

category_delivery.sort_values(
    by="late_delivery_rate",
    ascending=False
).head(15)

## SECTION 4 — Funnel Analysis

In [0]:
#Funnel Counts
total_orders = df["order_id"].nunique()

approved_orders = df[
    df["order_approved_at"].notnull()
]["order_id"].nunique()

shipped_orders = df[
    df["order_delivered_carrier_date"].notnull()
]["order_id"].nunique()

delivered_orders = df[
    df["order_delivered_customer_date"].notnull()
]["order_id"].nunique()

funnel_df = pd.DataFrame({
    "stage":[
        "Placed",
        "Approved",
        "Shipped",
        "Delivered"
    ],
    "orders":[
        total_orders,
        approved_orders,
        shipped_orders,
        delivered_orders
    ]
})

funnel_df

In [0]:
funnel_df["conversion_pct"] = (
    funnel_df["orders"] /
    funnel_df["orders"].iloc[0]
) * 100

funnel_df

## SECTION 5 — Category Intelligence

In [0]:
# Category Summary
category_summary = df.groupby(
    "product_category_name_english"
).agg({
    "payment_value":"sum",
    "review_score":"mean",
    "late_delivery_flag":"mean",
    "order_id":"nunique"
}).reset_index()

# Rename Columns
category_summary.columns = [
    "product_category_name_english",
    "total_revenue",
    "avg_review_score",
    "late_delivery_rate",
    "total_orders"
]

# Add Cancellation Rate
cancel_df = df.groupby(
    "product_category_name_english"
).apply(
    lambda x: (
        x["order_status"] == "canceled"
    ).mean() * 100
).reset_index(name="cancellation_rate")

# Merge
category_summary = category_summary.merge(
    cancel_df,
    on="product_category_name_english",
    how="left"
)

# Filter Reliable Categories
category_summary = category_summary[
    category_summary["total_orders"] > 100
]

# Sort by Revenue
category_summary.sort_values(
    by="total_revenue",
    ascending=False
).head(15)

## SECTION 6 — AI-Assisted Review Intelligence

In [0]:
negative_reviews = df[
    (df["review_score"] <= 2) &
    (df["review_comment_message"].notnull())
]

# Keep Relevant Columns
negative_reviews = negative_reviews[[
    "review_comment_message",
    "review_score",
    "product_category_name_english",
    "late_delivery_flag"
]]

negative_reviews.sample(10)

In [0]:
negative_reviews.to_csv("negative_reviews.csv", index=False)

In [0]:
from openai import OpenAI

client = OpenAI(
    api_key= "YOUR_API_KEY"
)

In [0]:
def classify_review(review):

    prompt = f"""
    Classify this ecommerce complaint into ONE category:

    - Delivery Delay
    - Product Quality
    - Damaged Product
    - Missing Item
    - Refund/Payment Issue
    - Wrong Product
    - Customer Service
    - Other

    Review:
    {review}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content

In [0]:
negative_reviews["complaint_theme"] = (
    negative_reviews["review_comment_message"]
    .head(100)
    .apply(classify_review)
)

In [0]:
negative_reviews = pd.read_csv(
    "negative_reviews_classified.csv"
)

theme_summary = negative_reviews.groupby(
    "complaint_theme"
).size().reset_index(name="reviews")

theme_summary.sort_values(
    by="reviews",
    ascending=False
)